# P2-07, P2-08 and P2-09
Run the cells in order on a CPU runtime. The inspection checks all audio headers and may take several hours on Drive. See docs/p2_07_09_execution.md for recovery and interpretation.


In [1]:
from google.colab import drive
drive.mount("/content/drive")
from pathlib import Path
import sys, os, json, subprocess
PROJECT_ROOT = Path("/content/drive/MyDrive/PAI&AIES Project")
assert (PROJECT_ROOT / "pyproject.toml").is_file(), "Set PROJECT_ROOT to your project"
os.environ["PYTHONPATH"] = str(PROJECT_ROOT / "src")
sys.path.insert(0, str(PROJECT_ROOT / "src"))
SNAPSHOT = PROJECT_ROOT / "data/manifests/phase2_raw/20261101T170035Z-7aac9dc154ad"
PAIRING = PROJECT_ROOT / "data/manifests/phase2_pairs/20261101T211532Z-ca15b910a412"
RESUME_P207 = "20261102T070051Z-4de313967107"  # Set the printed run ID after an interrupted run.


Mounted at /content/drive


## Dependencies and fixture checks
The cell stops if a required fixture check is skipped. No research audio is read here.


In [2]:
import yaml, pyarrow, soundfile, numpy
from echoproof.phase2_config import load_phase2_config
from echoproof.data.phase2_quality import inspect, reconcile, capacity
CONFIG = load_phase2_config(PROJECT_ROOT / "configs/phase2.yaml", PROJECT_ROOT)
check = subprocess.run([sys.executable,"-m","unittest","discover","-s",str(PROJECT_ROOT/"tests"),"-p","test_p2_07_09_quality.py","-v"],cwd=PROJECT_ROOT,text=True,capture_output=True)
print(check.stdout, check.stderr)
assert check.returncode == 0 and "skipped" not in (check.stdout + check.stderr).lower(), "Resolve failed or skipped checks"


P2-07 run ID: 20261103T145047Z-fcc231136ff5
P2-07 headers processed: 1/5
P2-07 headers processed: 2/5
P2-07 run ID: 20261103T145047Z-fcc231136ff5
P2-07 headers processed: 3/5
P2-07 headers processed: 4/5
P2-07 headers processed: 5/5
 test_corrupt_and_nonfinite (test_p2_07_09_quality.AudioQualityTest.test_corrupt_and_nonfinite) ... ok
test_valid_stereo_and_resource_limit (test_p2_07_09_quality.AudioQualityTest.test_valid_stereo_and_resource_limit) ... ok
test_capacity_unknown_and_reserve (test_p2_07_09_quality.QualityTest.test_capacity_unknown_and_reserve) ... ok
test_inspection_resume_uses_committed_header_rows (test_p2_07_09_quality.QualityTest.test_inspection_resume_uses_committed_header_rows) ... ok
test_reconciliation_propagates_attention_and_unrun_stages (test_p2_07_09_quality.QualityTest.test_reconciliation_propagates_attention_and_unrun_stages) ... ok
test_truncated_riff_and_safe_paths (test_p2_07_09_quality.QualityTest.test_truncated_riff_and_safe_paths) ... ok

---------------

## P2-07: inspect headers and sampled audio
Save the printed run ID. A completed or attention-needed inspection produces inspection_summary.json and admission.json.


In [5]:
import hashlib
import json
from pathlib import Path

audio = Path(PROJECT_ROOT) / "dataset/tts/VITS/6437-66173-0053.wav"
inventory = (
    Path(PROJECT_ROOT)
    / "data/manifests/phase2/20261101T061027Z-34a4086c2077"
    / "file_inventory.jsonl"
)

with inventory.open(encoding="utf-8") as stream:
    matches = [
        json.loads(line)
        for line in stream
        if '"6437-66173-0053.wav"' in line
        and '"component":"vits"' in line
    ]

assert len(matches) == 1, f"Expected one VITS inventory entry; found {len(matches)}"

hasher = hashlib.sha256()
with audio.open("rb") as stream:
    for block in iter(lambda: stream.read(1024 * 1024), b""):
        hasher.update(block)

print("Inventory SHA-256:", matches[0]["sha256"])
print("Restored SHA-256:", hasher.hexdigest())
print("Match:", hasher.hexdigest() == matches[0]["sha256"])

Inventory SHA-256: 4eeca65680447650ee388263c05f0cac0af13fac582508aa36766554d188287d
Restored SHA-256: 4eeca65680447650ee388263c05f0cac0af13fac582508aa36766554d188287d
Match: True


In [ ]:
inspection = inspect(PROJECT_ROOT, CONFIG, SNAPSHOT, PAIRING, resume=RESUME_P207 or None)
INSPECTION = Path(inspection["output"])
print("P2-07 output:", INSPECTION)
print(json.dumps(inspection,indent=2))


P2-07 run ID: 20261102T070051Z-4de313967107
P2-07 headers processed: 99600/142695
P2-07 headers processed: 99700/142695
P2-07 headers processed: 99800/142695
P2-07 headers processed: 99900/142695
P2-07 headers processed: 100000/142695
P2-07 headers processed: 100100/142695
P2-07 headers processed: 100200/142695
P2-07 headers processed: 100300/142695
P2-07 headers processed: 100400/142695
P2-07 headers processed: 100500/142695
P2-07 headers processed: 100600/142695
P2-07 headers processed: 100700/142695
P2-07 headers processed: 100800/142695
P2-07 headers processed: 100900/142695
P2-07 headers processed: 101000/142695
P2-07 headers processed: 101100/142695
P2-07 headers processed: 101200/142695
P2-07 headers processed: 101300/142695
P2-07 headers processed: 101400/142695
P2-07 headers processed: 101500/142695
P2-07 headers processed: 101600/142695
P2-07 headers processed: 101700/142695
P2-07 headers processed: 101800/142695
P2-07 headers processed: 101900/142695
P2-07 headers processed:

## P2-08: reconcile counts and generate the source card
If reconnecting in a new runtime, set INSPECTION to the completed P2-07 run directory before running this cell.


In [ ]:
INSPECTION = PROJECT_ROOT / "data/manifests/phase2_inspection/20261102T070051Z-4de313967107"
reconciliation = reconcile(PROJECT_ROOT, CONFIG, SNAPSHOT, PAIRING, INSPECTION)
print("P2-08 output:",reconciliation["output"])
print(json.dumps(reconciliation,indent=2))


P2-08 output: /content/drive/MyDrive/PAI&AIES Project/reports/phase2/p2_08-20261103T042748Z-5a59b852096f
{
  "output": "/content/drive/MyDrive/PAI&AIES Project/reports/phase2/p2_08-20261103T042748Z-5a59b852096f",
  "task": "P2-08",
  "run_id": "20261103T042748Z-5a59b852096f",
  "status": "needs_attention",
  "components": {
    "genuine": {
      "files_hashed_inventory": 28539,
      "bytes_inventory": 11589325052,
      "candidate_audio_records": 28539,
      "candidate_class": "genuine",
      "candidate_generator": null,
      "language": "unknown",
      "headers": {
        "ok": 28539
      },
      "sample_decodes": {
        "ok": 20
      },
      "admission": "pending"
    },
    "xtts": {
      "files_hashed_inventory": 28539,
      "bytes_inventory": 9561750812,
      "candidate_audio_records": 28539,
      "candidate_class": "tts",
      "candidate_generator": "xtts",
      "language": "unknown",
      "headers": {
        "ok": 28539
      },
      "sample_decodes": {
  

## P2-09: capacity record
Enter current account values from Google Drive storage, in decimal GB. Use None for an unknown value; the output will remain pending. The scratch measurement is separate from Drive account quota. No acquisition is started.


In [ ]:
DRIVE_TOTAL_GB = 30
DRIVE_USED_GB = 20.52
DRIVE_FREE_GB = 30-20.52
REMAINING_WORK_HOURS = 110
# Set these only for a specific planned acquisition, otherwise keep zero.
PLANNED_DOWNLOAD_BYTES = 0
EXPANDED_BYTES = 0
STAGING_BYTES = 0
CACHE_BYTES = 0
budget = capacity(PROJECT_ROOT, drive_total_gb=DRIVE_TOTAL_GB, drive_used_gb=DRIVE_USED_GB, drive_free_gb=DRIVE_FREE_GB, remaining_hours=REMAINING_WORK_HOURS, scratch_path="/content", planned_download_bytes=PLANNED_DOWNLOAD_BYTES, expanded_bytes=EXPANDED_BYTES, staging_bytes=STAGING_BYTES, cache_bytes=CACHE_BYTES)
print("P2-09 output:",budget["output"])
print(json.dumps(budget,indent=2))


P2-09 output: /content/drive/MyDrive/PAI&AIES Project/reports/phase2/p2_09-20261103T043812Z-d9c51ca4fd78
{
  "output": "/content/drive/MyDrive/PAI&AIES Project/reports/phase2/p2_09-20261103T043812Z-d9c51ca4fd78",
  "task": "P2-09",
  "captured_utc": "2026-11-03T04:38:12.673925+00:00",
  "drive_free_gb_user_supplied": 9.48,
  "drive_total_gb_user_supplied": 30,
  "drive_used_gb_user_supplied": 20.52,
  "drive_units": "decimal GB",
  "scratch_path": "/content",
  "scratch_free_bytes": 115213950976,
  "remaining_hours_user_supplied": 110,
  "planned_download_bytes": 0,
  "expanded_bytes": 0,
  "staging_bytes": 0,
  "cache_bytes": 0,
  "reserve_fraction": 0.25,
  "required_additional_bytes_with_reserve": 0,
  "drive_capacity_check": "not_required_no_acquisition",
  "scratch_capacity_check": "not_required_no_acquisition",
  "work_time": "provided",
  "acquisition": "none_planned",
  "note": "Drive account quota must be provided separately; scratch free space is not Drive quota. Existing ori